# Titanic Survival Prediction

**Internship Project | CodTech IT Solutions**

### Objective
Predict whether a passenger would have **survived** the Titanic disaster, based on features like age, sex, passenger class, fare, and family size, using supervised machine learning (Classification).

### Dataset
`Titanic.csv` — contains 891 passengers with details like Age, Sex, Pclass (ticket class), Fare, SibSp (siblings/spouses aboard), Parch (parents/children aboard), Embarked (port), and Survived (0 = No, 1 = Yes).

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)

## 1. Load the Dataset

In [6]:
df = pd.read_csv("Titanic.csv")
print("Shape:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Titanic.csv'

## 2. Exploratory Data Analysis (EDA)

Let's understand the data before building the model.

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
# Overall survival rate
survival_rate = df['survived'].mean() * 100
print(f"Overall survival rate: {survival_rate:.1f}%")

plt.figure(figsize=(6,4))
sns.countplot(x='survived', data=df, palette=['#e74c3c', '#2ecc71'])
plt.title("Survival Count (0 = Died, 1 = Survived)")
plt.xlabel("Survived")
plt.ylabel("Number of Passengers")
plt.show()

In [ ]:
# Survival rate by Sex and Class
fig, axes = plt.subplots(1, 2, figsize=(12,5))

sns.barplot(x='sex', y='survived', data=df, ax=axes[0], palette='Set2')
axes[0].set_title("Survival Rate by Gender")
axes[0].set_ylabel("Survival Rate")

sns.barplot(x='pclass', y='survived', data=df, ax=axes[1], palette='Set2')
axes[1].set_title("Survival Rate by Passenger Class")
axes[1].set_ylabel("Survival Rate")

plt.tight_layout()
plt.show()

In [ ]:
# Age distribution of survivors vs non-survivors
plt.figure(figsize=(8,5))
sns.histplot(data=df, x='age', hue='survived', bins=30, kde=True, palette=['#e74c3c', '#2ecc71'])
plt.title("Age Distribution: Survived vs Did Not Survive")
plt.xlabel("Age")
plt.show()

## 3. Data Preprocessing

In [ ]:
# Work on a copy so the original stays clean
data = df.copy()

# Fill missing Age with median age
data['age'] = data['age'].fillna(data['age'].median())

# Fill missing Embarked/embark_town with the most common port
data['embark_town'] = data['embark_town'].fillna(data['embark_town'].mode()[0])

# Drop columns that are duplicates of other columns or not useful for prediction
data = data.drop(columns=['deck', 'alive', 'class', 'who', 'adult_male', 'embarked', 'alone'])

# Create a family size feature
data['family_size'] = data['sibsp'] + data['parch'] + 1

data.isnull().sum()

In [ ]:
# Encode categorical columns to numbers
le_sex = LabelEncoder()
data['sex_encoded'] = le_sex.fit_transform(data['sex'])  # male=1, female=0 (alphabetical)

le_embark = LabelEncoder()
data['embark_encoded'] = le_embark.fit_transform(data['embark_town'])

data[['sex', 'sex_encoded', 'embark_town', 'embark_encoded']].drop_duplicates()

In [ ]:
features = ['pclass', 'sex_encoded', 'age', 'sibsp', 'parch', 'fare', 'embark_encoded', 'family_size']
X = data[features]
y = data['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 4. Training the Model

We use a **Random Forest Classifier** — it usually performs well on this dataset and also gives us feature importance.

In [ ]:
model = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy*100:.2f}%")

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Did not survive', 'Survived'],
            yticklabels=['Did not survive', 'Survived'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print(classification_report(y_test, y_pred, target_names=['Did not survive', 'Survived']))

## 6. Feature Importance

Which features mattered most to the model?

In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title("Feature Importance")
plt.xlabel("Importance")
plt.show()

importances

## 7. Business / Historical Insights

- **Gender mattered most**: Women had a much higher survival rate than men ("women and children first").
- **Passenger class mattered**: 1st class passengers survived at a much higher rate than 3rd class — likely due to cabin location and lifeboat access.
- **Fare and Class are related**: Higher fare (often tied to class) correlates with higher survival.
- **Family size** had a moderate effect — travelling completely alone or in a very large family both slightly lowered survival chances compared to small families.

## 8. Interactive Prediction Demo

This is the part you'll actually **run live** in front of your teacher. Run the cell below, then enter passenger details when it asks — the model will predict Survived / Did Not Survive with a probability.

In [ ]:
def predict_survival():
    print("Enter passenger details to predict survival:\n")

    pclass = int(input("Passenger Class (1 = First, 2 = Second, 3 = Third): "))
    sex = input("Sex (male/female): ").strip().lower()
    age = float(input("Age: "))
    sibsp = int(input("Number of Siblings/Spouses aboard: "))
    parch = int(input("Number of Parents/Children aboard: "))
    fare = float(input("Fare paid (e.g. 32.5): "))
    embark_town = input("Embarkation Town (Southampton/Cherbourg/Queenstown): ").strip()

    sex_encoded = le_sex.transform([sex])[0]
    embark_encoded = le_embark.transform([embark_town])[0]
    family_size = sibsp + parch + 1

    input_df = pd.DataFrame([[pclass, sex_encoded, age, sibsp, parch, fare, embark_encoded, family_size]],
                            columns=features)

    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0][1]

    print("\n--- Prediction Result ---")
    if prediction == 1:
        print(f"SURVIVED (probability: {probability*100:.1f}%)")
    else:
        print(f"DID NOT SURVIVE (probability of survival: {probability*100:.1f}%)")

# Call the function to try it
predict_survival()

**Quick test values you can use during the demo** (so you don't have to think of numbers live):
- A 1st class woman, age 28, no family, fare 100, embarked from Cherbourg → very likely **Survived**
- A 3rd class man, age 25, no family, fare 8, embarked from Southampton → very likely **Did not survive**